# DeepAR forecast demo

Train a probabilistic DeepAR model on synthetic autoregressive data using the current task API.
The task minimizes Gaussian negative log likelihood and gradually reduces teacher forcing.
At inference, `generate()` samples complete trajectories and returns their mean and sample paths.
Training duration and random seeds affect the resulting metrics; this notebook is a workflow example.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from tfts import AutoConfig, AutoModel, AutoModelForForecasting, ForecastTaskConfig, GenerationConfig

tf.keras.utils.set_random_seed(42)
ENCODER_LENGTH = 60
PREDICTION_LENGTH = 20
N_SAMPLES = 100
EPOCHS = 40
BATCH_SIZE = 128

## 1. Data
Generate the synthetic AR data (quadratic trend + seasonality + noise, 100 series x 400 steps) using the same `generate_ar_data(...)` call and seed as the tutorial.


In [ ]:
from tfts.data.ar import generate_ar_data

data = generate_ar_data(seasonality=10.0, timesteps=400, n_series=100, seed=42)
print(data.head())
print(
    "rows:",
    len(data),
    "| series:",
    data.series.nunique(),
    "| time range:",
    data.time_idx.min(),
    "-",
    data.time_idx.max(),
)

## 2. Windowing and normalization

Hold out the final forecast horizon of each series. Fit normalization on training rows only.
The model receives `past_values` and static series IDs; the task handles teacher feedback from labels.


In [ ]:
from tfts.data.ar import ARDeepARPreprocessor

# global target normalizer fit on the training portion (mirrors PF EncoderNormalizer)
cutoff = data.time_idx.max() - PREDICTION_LENGTH
train_vals = data.loc[data.time_idx <= cutoff, "value"]
mean, std = float(train_vals.mean()), float(train_vals.std())
print("normalizer  mean:", round(mean, 4), " std:", round(std, 4))

proc = ARDeepARPreprocessor(
    data, encoder_length=ENCODER_LENGTH, prediction_length=PREDICTION_LENGTH, mean=mean, std=std
)
train_batch = proc.train()  # sliding windows fully in time_idx <= training_cutoff
val_batch = proc.validation()  # one forecast per series starting at training_cutoff+1

print("train x:", train_batch.x.shape, "  y:", train_batch.y.shape)
print("train decoder_feature:", train_batch.decoder_feature.shape, "  static:", train_batch.static.shape)
print("val   x:", val_batch.x.shape, "  y:", val_batch.y.shape)

## 3. Build the model
Use canonical target histories and static series IDs. The task owns Gaussian NLL and scheduled sampling.

In [ ]:
cfg = AutoConfig.for_model("deep_ar")
cfg.n_series = proc.n_series
steps_per_epoch = int(np.ceil(len(train_batch.x) / BATCH_SIZE))
forecast_model = AutoModelForForecasting.from_config(
    cfg,
    task_config=ForecastTaskConfig(
        output_chunk_length=PREDICTION_LENGTH,
        head="native",
        feedback_sampler="sample",
        teacher_probability=1.0,
        teacher_final_probability=0.2,
        teacher_decay_steps=EPOCHS * steps_per_epoch,
    ),
)

## 4. Train with scheduled sampling
The task's default objective is Gaussian negative log likelihood. Teacher probability decays from 1.0 to 0.2 over training.

In [ ]:
train_inputs = {"past_values": train_batch.x, "static_categorical_features": train_batch.static}
val_inputs = {"past_values": val_batch.x, "static_categorical_features": val_batch.static}
train_ds = tf.data.Dataset.from_tensor_slices((train_inputs, train_batch.y)).shuffle(4096, seed=42).batch(BATCH_SIZE)
val_ds = tf.data.Dataset.from_tensor_slices((val_inputs, val_batch.y)).batch(BATCH_SIZE)
forecast_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=0.1))
history = forecast_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=2)

## 5. Evaluate ancestral samples

`generate()` draws complete trajectories, feeding each sampled value into the next step.
Inverse-transform the mean predictions and samples to report and plot original units.


In [ ]:
gen = forecast_model.generate(
    {"past_values": val_batch.x, "static_categorical_features": val_batch.static},
    config=GenerationConfig(mode="native", num_samples=100, seed=0),
)
preds = proc.inverse_transform(gen.predictions.numpy()[..., 0])
actual = val_batch.target_original[..., 0]
err = actual - preds
print("MAE:", float(np.abs(err).mean()))
print("MSE:", float(np.mean(err**2)))

## 6. Plot a forecast


In [ ]:
s = 0  # pick a series
window = proc.inverse_transform(val_batch.x[s, :, 0])  # normalized history (encoder window)
fut = np.arange(PREDICTION_LENGTH)

plt.figure(figsize=(10, 4))
plt.plot(np.arange(-ENCODER_LENGTH, 0), window, label="history")
plt.plot(fut, actual[s], label="actual", marker="o")
plt.plot(fut, preds[s], label="forecast (mean of 100 sampled paths)", marker="x")
samples = proc.inverse_transform(gen.samples.numpy()[s, :, :, 0])  # (num_samples, pred_len)
lo, hi = np.percentile(samples, 10, axis=0), np.percentile(samples, 90, axis=0)
plt.fill_between(fut, lo, hi, alpha=0.15, color="tab:blue", label="10-90% sample band")
plt.axvline(0, color="gray", ls="--")
plt.xlabel("time (relative to forecast start)")
plt.ylabel("value")
plt.legend()
plt.title(f"Series {s}: {PREDICTION_LENGTH}-step DeepAR forecast ")
plt.tight_layout()
plt.show()

## 7. Save, reload, and run inference
Use `generate()` for ancestral-sampling inference on the restored model.


In [ ]:
_ = forecast_model(
    {
        "past_values": val_batch.x[:1],
        "future_values": val_batch.y[:1],
        "static_categorical_features": val_batch.static[:1],
    }
)
forecast_model.save_pretrained("./deepar_model")
restored_model = AutoModel.from_pretrained(
    "./deepar_model",
    sample_batch={
        "past_values": val_batch.x[:1],
        "future_values": val_batch.y[:1],
        "static_categorical_features": val_batch.static[:1],
    },
)

restored_gen = restored_model.generate(
    {"past_values": val_batch.x, "static_categorical_features": val_batch.static},
    config=GenerationConfig(mode="native", num_samples=100, seed=0),
)
restored_preds = proc.inverse_transform(restored_gen.predictions.numpy()[..., 0])
print("Restored prediction shape:", restored_preds.shape)
print("Restored MAE:", float(np.abs(actual - restored_preds).mean()))